# This notebook tests the pipeline

In [3]:
!pip install psycopg2-binary sqlalchemy python-dotenv pandas numpy sentence-transformers pgvector tqdm

In [10]:
from google.colab import userdata

DATABASE_URL = userdata.get("DATABASE_URL")

DATABASE_URL = DATABASE_URL.strip().replace("\ufeff", "")

print(repr(DATABASE_URL[-30:]))

'ler.supabase.com:5432/postgres'


In [11]:
from sqlalchemy import create_engine, text

engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    result = conn.execute(text("SELECT now();"))
    print("Connected:", result.scalar())

Connected: 2026-05-01 13:39:23.910597+00:00


In [12]:
with engine.begin() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS pgcrypto;"))
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector;"))
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis;"))

print("Extensions enabled")

Extensions enabled


In [13]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("mock_recommender_dataset_v2.zip")
EXTRACT_DIR = Path("mock_recommender_dataset_v2")

if not ZIP_PATH.exists():
    raise FileNotFoundError("mock_recommender_dataset_v2.zip not found")

if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)

Extracted to: mock_recommender_dataset_v2


In [27]:
import json
import pandas as pd

def load_json(name):
    with open(EXTRACT_DIR / name, "r", encoding="utf-8") as f:
        return json.load(f)

users = load_json("users.json")
user_preferences = load_json("user_preferences.json")
groups = load_json("groups.json")
group_members = load_json("group_members.json")
restaurants = load_json("restaurants.json")
menus = load_json("menus.json")
reviews = load_json("reviews.json")
match_histories = load_json("match_histories.json")
tag_catalogs = load_json("tag_catalogs.json")
menu_dictionary = load_json("menu_dictionary.json")
test_cases = load_json("recommendation_test_cases.json")

print("users:", len(users))
print("user_preferences:", len(user_preferences))
print("groups:", len(groups))
print("group_members:", len(group_members))
print("restaurants:", len(restaurants))
print("menus:", len(menus))
print("reviews:", len(reviews))
print("match_histories:", len(match_histories))
print("tag_catalogs:", len(tag_catalogs))
print("menu_dictionary:", len(menu_dictionary))
print("test_cases:", len(test_cases))

users: 20
user_preferences: 20
groups: 5
group_members: 15
restaurants: 220
menus: 678
reviews: 180
match_histories: 500
tag_catalogs: 141
menu_dictionary: 31
test_cases: 10


In [28]:
!pip install -q sentence-transformers tqdm pgvector

In [29]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

def embed_texts(texts, batch_size=64):
    return model.encode(
        [t or "" for t in texts],
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    ).tolist()

menu_embeddings_th = embed_texts([m.get("embedding_text_th", "") for m in menus])
menu_embeddings_en = embed_texts([m.get("embedding_text_en", "") for m in menus])

restaurant_embeddings_th = embed_texts([r.get("embedding_text_th", "") for r in restaurants])
restaurant_embeddings_en = embed_texts([r.get("embedding_text_en", "") for r in restaurants])

for i, m in enumerate(menus):
    m["embedding_th"] = menu_embeddings_th[i]
    m["embedding_en"] = menu_embeddings_en[i]

for i, r in enumerate(restaurants):
    r["embedding_th"] = restaurant_embeddings_th[i]
    r["embedding_en"] = restaurant_embeddings_en[i]

print("Embeddings generated.")
print("Menu embedding dim:", len(menus[0]["embedding_en"]))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings generated.
Menu embedding dim: 384


In [30]:
from sqlalchemy import text
from tqdm import tqdm

target_tables = [
    "users",
    "user_preferences",
    "groups",
    "group_members",
    "restaurants",
    "menus",
    "reviews",
    "match_histories",
    "tag_catalogs",
    "menu_dictionary",
]

def get_table_columns(table_name):
    with engine.connect() as conn:
        rows = conn.execute(text("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = :table_name
        """), {"table_name": table_name}).mappings().all()
    return {r["column_name"] for r in rows}

table_columns = {table: get_table_columns(table) for table in target_tables}

for table, cols in table_columns.items():
    print(table, len(cols))

users 5
user_preferences 16
groups 4
group_members 3
restaurants 30
menus 21
reviews 7
match_histories 10
tag_catalogs 8
menu_dictionary 11


In [33]:
import json
from sqlalchemy import text
from tqdm import tqdm

JSONB_COLUMNS = {
    "user_preferences": {"learned_preferences"},
    "restaurants": {"opening_hours"},
    "match_histories": {"query_context", "score_breakdown"},
}

VECTOR_COLUMNS = {
    "restaurants": {"embedding_th", "embedding_en"},
    "menus": {"embedding_th", "embedding_en"},
}

def vector_to_pgvector(v):
    # pgvector accepts string like '[0.1,0.2,0.3]'
    if v is None:
        return None
    return "[" + ",".join(str(float(x)) for x in v) + "]"

def clean_row_for_table(row, table_name):
    row = dict(row)

    # Mock-only helper fields
    row.pop("embedding_text_th", None)
    row.pop("embedding_text_en", None)
    row.pop("location_wkt", None)

    allowed = table_columns[table_name]
    row = {k: v for k, v in row.items() if k in allowed}

    # Convert JSONB dict/list values
    for col in JSONB_COLUMNS.get(table_name, set()):
        if col in row and row[col] is not None:
            row[col] = json.dumps(row[col], ensure_ascii=False)

    # Convert vector lists to pgvector string
    for col in VECTOR_COLUMNS.get(table_name, set()):
        if col in row and row[col] is not None:
            row[col] = vector_to_pgvector(row[col])

    return row


def insert_rows_safe(table_name, rows):
    if not rows:
        return

    with engine.begin() as conn:
        for raw_row in tqdm(rows, desc=f"inserting {table_name}"):
            row = clean_row_for_table(raw_row, table_name)

            cols = list(row.keys())
            placeholders = []

            for c in cols:
                if c in JSONB_COLUMNS.get(table_name, set()):
                    placeholders.append(f"CAST(:{c} AS jsonb)")
                elif c in VECTOR_COLUMNS.get(table_name, set()):
                    placeholders.append(f"CAST(:{c} AS vector)")
                else:
                    placeholders.append(f":{c}")

            sql = f"""
            INSERT INTO {table_name} ({", ".join(cols)})
            VALUES ({", ".join(placeholders)})
            ON CONFLICT DO NOTHING
            """

            conn.execute(text(sql), row)

    print(f"Inserted/ignored {len(rows)} rows into {table_name}")


def insert_restaurants_safe(restaurants):
    with engine.begin() as conn:
        for raw_row in tqdm(restaurants, desc="inserting restaurants"):
            location_wkt = raw_row.get("location_wkt")
            row = clean_row_for_table(raw_row, "restaurants")

            cols = list(row.keys())
            placeholders = []

            for c in cols:
                if c in JSONB_COLUMNS.get("restaurants", set()):
                    placeholders.append(f"CAST(:{c} AS jsonb)")
                elif c in VECTOR_COLUMNS.get("restaurants", set()):
                    placeholders.append(f"CAST(:{c} AS vector)")
                else:
                    placeholders.append(f":{c}")

            if "location" in table_columns["restaurants"] and location_wkt:
                cols.append("location")
                placeholders.append("ST_GeogFromText(:location_wkt)")
                row["location_wkt"] = location_wkt

            sql = f"""
            INSERT INTO restaurants ({", ".join(cols)})
            VALUES ({", ".join(placeholders)})
            ON CONFLICT DO NOTHING
            """

            conn.execute(text(sql), row)

    print(f"Inserted/ignored {len(restaurants)} restaurants")

In [34]:
insert_rows_safe("users", users)
insert_rows_safe("user_preferences", user_preferences)
insert_rows_safe("groups", groups)
insert_rows_safe("group_members", group_members)

# These may overlap conceptually with seed data, so ON CONFLICT DO NOTHING is important.
insert_rows_safe("tag_catalogs", tag_catalogs)
insert_rows_safe("menu_dictionary", menu_dictionary)

insert_restaurants_safe(restaurants)

insert_rows_safe("menus", menus)
insert_rows_safe("reviews", reviews)
insert_rows_safe("match_histories", match_histories)

print("Mock v2 data inserted safely.")

inserting users: 100%|██████████| 20/20 [00:01<00:00, 12.40it/s]


Inserted/ignored 20 rows into users


inserting user_preferences: 100%|██████████| 20/20 [00:01<00:00, 12.20it/s]


Inserted/ignored 20 rows into user_preferences


inserting groups: 100%|██████████| 5/5 [00:00<00:00, 10.83it/s]


...

Inserted/ignored 678 rows into menus


inserting reviews: 100%|██████████| 180/180 [00:13<00:00, 12.92it/s]


Inserted/ignored 180 rows into reviews


inserting match_histories: 100%|██████████| 500/500 [00:38<00:00, 12.94it/s]

Inserted/ignored 500 rows into match_histories
Mock v2 data inserted safely.


In [35]:
counts = []

with engine.connect() as conn:
    for table in target_tables:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table};")).scalar()
        counts.append({"table": table, "row_count": count})

pd.DataFrame(counts)

,table,row_count
0,users,22
1,user_preferences,21
2,groups,5
3,group_members,15
4,restaurants,222
5,menus,681
6,reviews,180
7,match_histories,500
8,tag_catalogs,160
9,menu_dictionary,33


Test

In [36]:
case = test_cases[0]
case

{'id': 'case_001',
 'name': 'No pork spicy local',
 'user_id': '00000000-0000-0000-0000-000000000001',
 'group_id': None,
 'query_en': 'I want spicy local Thai food, not touristy, no pork',
 'query_th': 'อยากกินอาหารไทยรสจัด local ไม่เอาหมู',
 'user_location': {'latitude': 13.7799, 'longitude': 100.5447},
 'hard_constraints': {'dietary_restrictions': ['no_pork'],
  'allergies': [],
  'max_distance_km': 3.0,
  'budget_max': 300},
 'expected': {'must_exclude_menu_tags': ['contains_pork'],
  'prefer_tags': ['spicy', 'isaan_food', 'local_hidden_gem']}}

In [37]:
query_vec = model.encode(
    [case["query_en"]],
    normalize_embeddings=True
)[0].tolist()

params = {
    "query_vec": str(query_vec),
    "user_lng": case["user_location"]["longitude"],
    "user_lat": case["user_location"]["latitude"],
    "max_distance_m": case["hard_constraints"]["max_distance_km"] * 1000,
    "budget_max": case["hard_constraints"]["budget_max"],
}

sql = """
WITH safe_menu_matches AS (
    SELECT
        m.id AS menu_id,
        m.restaurant_id,
        m.name_en AS menu_name_en,
        m.name_th AS menu_name_th,
        m.food_tags,
        m.taste_tags,
        m.allergen_tags,
        m.dietary_tags,
        m.price,
        1 - (m.embedding_en <=> CAST(:query_vec AS vector)) AS menu_similarity
    FROM menus m
    JOIN restaurants r ON r.id = m.restaurant_id
    WHERE
        r.is_open = true
        AND m.confirmed_by_restaurant = true
        AND m.price <= :budget_max
        AND 'contains_pork' <> ALL(m.dietary_tags)
),
nearby_restaurants AS (
    SELECT
        r.*,
        ST_Distance(
            r.location,
            ST_SetSRID(ST_MakePoint(:user_lng, :user_lat), 4326)::geography
        ) / 1000.0 AS distance_km
    FROM restaurants r
    WHERE
        r.is_open = true
        AND ST_DWithin(
            r.location,
            ST_SetSRID(ST_MakePoint(:user_lng, :user_lat), 4326)::geography,
            :max_distance_m
        )
),
restaurant_candidates AS (
    SELECT
        r.id AS restaurant_id,
        r.name_en,
        r.name_th,
        r.google_maps_url,
        r.service_tags,
        r.place_context_tags,
        r.atmosphere_tags,
        r.restaurant_dietary_tags,
        r.distance_km,
        MAX(s.menu_similarity) AS best_menu_similarity,
        COUNT(*) AS matching_menu_count,
        ARRAY_AGG(s.menu_name_en ORDER BY s.menu_similarity DESC) AS matched_menus
    FROM safe_menu_matches s
    JOIN nearby_restaurants r ON r.id = s.restaurant_id
    GROUP BY
        r.id,
        r.name_en,
        r.name_th,
        r.google_maps_url,
        r.service_tags,
        r.place_context_tags,
        r.atmosphere_tags,
        r.restaurant_dietary_tags,
        r.distance_km
)
SELECT
    *,
    (
        45 * best_menu_similarity
        + 15 * LEAST(matching_menu_count, 3) / 3.0
        + 15 * GREATEST(0, 1 - distance_km / (:max_distance_m / 1000.0))
        + 10 * CASE WHEN 'local_hidden_gem' = ANY(place_context_tags) THEN 1 ELSE 0 END
        + 5  * CASE WHEN 'family_friendly' = ANY(atmosphere_tags) THEN 1 ELSE 0 END
    ) AS final_score
FROM restaurant_candidates
ORDER BY final_score DESC
LIMIT 10;
"""

with engine.connect() as conn:
    rows = conn.execute(text(sql), params).mappings().all()

result = pd.DataFrame(rows)
result

,atmosphere_tags,best_menu_similarity,distance_km,final_score,google_maps_url,matched_menus,matching_menu_count,name_en,name_th,place_context_tags,restaurant_dietary_tags,restaurant_id,service_tags
0,"[local, old_school, family_run, casual]",0.520197,0.776431,59.526687,"https://maps.google.com/?q=13.776740,100.538289","[Som Tam Pu Pla Ra, Grilled Chicken, Sticky Rice]",3,Ari Local Isaan 9,อีสานบ้านๆ อารีย์ 9,"[walkable, local_hidden_gem, near_bts]",[],10000000-0000-0000-0000-000000000009,"[quick_service, cash_only]"
1,"[local, casual, old_school]",0.498924,0.710676,58.898182,"https://maps.google.com/?q=13.774434,100.541248","[Yellow Curry with Sea Bass, Shrimp Paste Chil...",3,Victory Monument Southern Seafood Curry 26,แกงใต้บ้านเรา อนุสาวรีย์ชัย 26,"[local_hidden_gem, near_bts, local_market]","[no_pork_friendly, vegetarian_friendly]",10000000-0000-0000-0000-000000000026,"[quick_service, english_limited]"
2,"[local, family_run, casual]",0.520197,1.669069,55.063500,"https://maps.google.com/?q=13.786208,100.530679","[Som Tam Pu Pla Ra, Grilled Chicken, Sticky Rice]",3,Ari Local Isaan 64,อีสานบ้านๆ อารีย์ 64,"[walkable, local_hidden_gem, near_bts]",[],10000000-0000-0000-0000-000000000064,"[quick_service, cash_only]"
3,"[local, hidden_gem, family_run, casual]",0.490474,1.529180,54.425416,"https://maps.google.com/?q=13.766293,100.547182","[Thai Papaya Salad with Salted Egg, Grilled Ti...",3,Ari No-Pork Isaan 15,อีสานไก่ย่าง อารีย์ 15,"[good_for_group, walkable, local_hidden_gem, n...",[no_pork_friendly],10000000-0000-0000-0000-000000000015,"[quick_service, parking_available]"
4,"[local, family_run, casual]",0.520197,1.961572,53.600985,"https://maps.google.com/?q=13.796451,100.538196","[Som Tam Pu Pla Ra, Grilled Chicken, Sticky Rice]",3,Ari Local Isaan 80,อีสานบ้านๆ อารีย์ 80,"[walkable, local_hidden_gem, near_bts]",[],10000000-0000-0000-0000-000000000080,"[quick_service, cash_only, friendly_staff]"
5,"[busy, photo_spot, modern, family_run]",0.541066,0.302445,52.835746,"https://maps.google.com/?q=13.778698,100.547212","[Tom Yum Goong, Shrimp Pad Thai, Chicken Green...",3,Victory Monument Thai Food Cafe 63,ไทยฟู้ดคาเฟ่ อนุสาวรีย์ชัย 63,"[near_bts, local_market]",[vegetarian_friendly],10000000-0000-0000-0000-000000000063,"[english_menu_available, friendly_staff, parki..."
6,"[local, casual, family_friendly, family_run, q...",0.434414,1.364923,52.724022,"https://maps.google.com/?q=13.776018,100.532719","[Roti with Chicken Curry, Beef Massaman Curry]",2,Victory Monument Thai Muslim Kitchen 51,ครัวมุสลิม อนุสาวรีย์ชัย 51,"[good_for_group, local_hidden_gem, near_bts, l...","[halal_certified, no_pork_friendly, vegetarian...",10000000-0000-0000-0000-000000000051,"[parking_available, card_accepted]"
7,"[local, quiet, family_run, casual]",0.520197,0.304105,51.888319,"https://maps.google.com/?q=13.777179,100.544302","[Som Tam Pu Pla Ra, Grilled Chicken, Sticky Rice]",3,Victory Monument Local Isaan 126,อีสานบ้านๆ อนุสาวรีย์ชัย 126,"[good_for_group, near_bts, local_market]",[],10000000-0000-0000-0000-000000000126,"[quick_service, cash_only, friendly_staff]"
8,"[local, busy, quiet, family_friendly]",0.358765,0.187064,50.209090,"https://maps.google.com/?q=13.778242,100.544361",[Baked Prawns with Glass Noodles],1,Ari Local Seafood 94,ซีฟู้ดชุมชน อารีย์ 94,"[walkable, local_hidden_gem, near_bts]",[no_pork_friendly],10000000-0000-0000-0000-000000000094,"[quick_service, friendly_staff, parking_availa..."
9,"[casual, quiet, local]",0.397204,0.563985,50.054252,"https://maps.google.com/?q=13.776254,100.541055","[Vegetable Stew, Tofu Basil Stir Fry]",2,Ari Kind Veg Thai 123,ครัวเจใจดี อารีย์ 123,"[walkable, local_hidden_gem, near_bts]","[vegan_friendly, vegetarian_friendly, no_pork_...",10000000-0000-0000-0000-000000000123,"[english_menu_available, card_accepted]"


In [38]:
if not result.empty:
    print(result[["name_en", "distance_km", "best_menu_similarity", "matching_menu_count", "final_score", "matched_menus"]])
else:
    print("No recommendations found.")

                                      name_en  distance_km  \
0                           Ari Local Isaan 9     0.776431   
1  Victory Monument Southern Seafood Curry 26     0.710676   
2                          Ari Local Isaan 64     1.669069   
3                        Ari No-Pork Isaan 15     1.529180   
4                          Ari Local Isaan 80     1.961572   
5          Victory Monument Thai Food Cafe 63     0.302445   
6     Victory Monument Thai Muslim Kitchen 51     1.364923   
7            Victory Monument Local Isaan 126     0.304105   
8                        Ari Local Seafood 94     0.187064   
9                       Ari Kind Veg Thai 123     0.563985   

   best_menu_similarity  matching_menu_count  final_score  \
0              0.520197                    3    59.526687   
1              0.498924                    3    58.898182   
2              0.520197                    3    55.063500   
3              0.490474                    3    54.425416   
4           

menu first

In [40]:
def recommend_menu_first(
    query,
    user_lat,
    user_lng,
    max_distance_km=3.0,
    budget_max=300,
    dietary_restrictions=None,
    allergies=None,
    service_tags=None,
    language="en",
    limit=10,
):
    dietary_restrictions = dietary_restrictions or []
    allergies = allergies or []
    service_tags = service_tags or []

    query_vec = model.encode([query], normalize_embeddings=True)[0].tolist()
    embedding_col = "embedding_en" if language == "en" else "embedding_th"

    # Hard safety rules:
    # - only trust menu safety tags when confirmed_by_restaurant = true
    # - for allergy/diet hard constraints, filter at menu level
    no_pork = "no_pork" in dietary_restrictions or "pork" in allergies
    vegetarian = "vegetarian" in dietary_restrictions
    halal_required = "halal_required" in dietary_restrictions

    allergy_filters = ""
    params = {
        "query_vec": str(query_vec),
        "user_lng": user_lng,
        "user_lat": user_lat,
        "max_distance_m": max_distance_km * 1000,
        "budget_max": budget_max,
        "limit": limit,
    }

    if allergies:
        allergy_filters += " AND NOT (m.allergen_tags && :allergies) "
        params["allergies"] = allergies

    dietary_filters = ""

    if no_pork:
        dietary_filters += " AND 'contains_pork' <> ALL(m.dietary_tags) "

    if vegetarian:
        dietary_filters += " AND 'vegetarian_possible' = ANY(m.dietary_tags) "

    restaurant_filters = ""

    if halal_required:
        restaurant_filters += " AND 'halal_certified' = ANY(r.restaurant_dietary_tags) "

    if service_tags:
        restaurant_filters += " AND r.service_tags @> :service_tags "
        params["service_tags"] = service_tags

    sql = f"""
    WITH safe_menu_matches AS (
        SELECT
            m.id AS menu_id,
            m.restaurant_id,
            m.name_en AS menu_name_en,
            m.name_th AS menu_name_th,
            m.food_tags,
            m.taste_tags,
            m.allergen_tags,
            m.dietary_tags,
            m.price,
            m.confirmed_by_restaurant,
            1 - (m.{embedding_col} <=> CAST(:query_vec AS vector)) AS menu_similarity
        FROM menus m
        JOIN restaurants r ON r.id = m.restaurant_id
        WHERE
            r.is_open = true
            AND m.confirmed_by_restaurant = true
            AND m.price <= :budget_max
            {allergy_filters}
            {dietary_filters}
    ),
    nearby_restaurants AS (
        SELECT
            r.*,
            ST_Distance(
                r.location,
                ST_SetSRID(ST_MakePoint(:user_lng, :user_lat), 4326)::geography
            ) / 1000.0 AS distance_km
        FROM restaurants r
        WHERE
            r.is_open = true
            AND ST_DWithin(
                r.location,
                ST_SetSRID(ST_MakePoint(:user_lng, :user_lat), 4326)::geography,
                :max_distance_m
            )
            {restaurant_filters}
    ),
    restaurant_candidates AS (
        SELECT
            r.id AS restaurant_id,
            r.name_en,
            r.name_th,
            r.google_maps_url,
            r.service_tags,
            r.place_context_tags,
            r.atmosphere_tags,
            r.restaurant_dietary_tags,
            r.price_min,
            r.price_max,
            r.distance_km,
            MAX(s.menu_similarity) AS best_menu_similarity,
            COUNT(*) AS matching_menu_count,
            ARRAY_AGG(s.menu_name_en ORDER BY s.menu_similarity DESC) AS matched_menus
        FROM safe_menu_matches s
        JOIN nearby_restaurants r ON r.id = s.restaurant_id
        GROUP BY
            r.id,
            r.name_en,
            r.name_th,
            r.google_maps_url,
            r.service_tags,
            r.place_context_tags,
            r.atmosphere_tags,
            r.restaurant_dietary_tags,
            r.price_min,
            r.price_max,
            r.distance_km
    )
    SELECT
        *,
        (
            45 * best_menu_similarity
            + 15 * LEAST(matching_menu_count, 3) / 3.0
            + 15 * GREATEST(0, 1 - distance_km / :max_distance_m * 1000)
            + 10 * CASE WHEN 'local_hidden_gem' = ANY(place_context_tags) THEN 1 ELSE 0 END
            + 10 * CASE WHEN 'parking_available' = ANY(service_tags) THEN 1 ELSE 0 END
            + 5  * CASE WHEN 'family_friendly' = ANY(atmosphere_tags) THEN 1 ELSE 0 END
        ) AS final_score
    FROM restaurant_candidates
    ORDER BY final_score DESC
    LIMIT :limit;
    """

    with engine.connect() as conn:
        rows = conn.execute(text(sql), params).mappings().all()

    return pd.DataFrame(rows)

In [41]:
case = test_cases[2]  # Halal group parking

halal_result = recommend_menu_first(
    query=case["query_en"],
    user_lat=case["user_location"]["latitude"],
    user_lng=case["user_location"]["longitude"],
    max_distance_km=case["hard_constraints"]["max_distance_km"],
    budget_max=case["hard_constraints"]["budget_max"],
    dietary_restrictions=case["hard_constraints"].get("dietary_restrictions", []),
    allergies=case["hard_constraints"].get("allergies", []),
    service_tags=case["hard_constraints"].get("service_tags", []),
    language="en",
    limit=10,
)

halal_result

,atmosphere_tags,best_menu_similarity,distance_km,final_score,google_maps_url,matched_menus,matching_menu_count,name_en,name_th,place_context_tags,price_max,price_min,restaurant_dietary_tags,restaurant_id,service_tags
0,"[local, casual, family_friendly]",0.624601,0.429117,82.034238,"https://maps.google.com/?q=13.755944,100.492175","[Roti with Chicken Curry, Beef Massaman Curry,...",3,Old Town Thai Muslim Kitchen 107,ครัวมุสลิม เมืองเก่า 107,"[temple_nearby, local_hidden_gem]",190.00,100.00,"[halal_certified, no_pork_friendly, vegetarian...",10000000-0000-0000-0000-000000000107,"[friendly_staff, parking_available, card_accep..."
1,"[local, old_school, casual, family_friendly]",0.624601,2.407499,77.088284,"https://maps.google.com/?q=13.738069,100.510661","[Roti with Chicken Curry, Beef Massaman Curry,...",3,Old Town Thai Muslim Kitchen 41,ครัวมุสลิม เมืองเก่า 41,"[good_for_group, temple_nearby, local_hidden_gem]",370.00,120.00,"[halal_certified, no_pork_friendly]",10000000-0000-0000-0000-000000000041,"[friendly_staff, parking_available, card_accep..."
2,"[local, casual, modern, family_friendly, famil...",0.624601,3.423909,69.547259,"https://maps.google.com/?q=13.749035,100.525460","[Roti with Chicken Curry, Beef Massaman Curry]",2,Ratchathewi Thai Muslim Kitchen 69,ครัวมุสลิม ราชเทวี 69,"[good_for_group, walkable, local_hidden_gem, n...",410.00,60.00,"[halal_certified, no_pork_friendly]",10000000-0000-0000-0000-000000000069,"[friendly_staff, parking_available, card_accep..."
3,"[local, quiet, casual, family_friendly]",0.624601,4.552866,66.724867,"https://maps.google.com/?q=13.720515,100.520486","[Roti with Chicken Curry, Beef Massaman Curry]",2,Silom Thai Muslim Kitchen 90,ครัวมุสลิม สีลม 90,"[business_area, local_hidden_gem, near_bts, go...",410.00,60.00,"[halal_certified, no_pork_friendly]",10000000-0000-0000-0000-000000000090,"[parking_available, card_accepted]"
4,"[local, casual, family_friendly, family_run, q...",0.624601,4.929791,65.782555,"https://maps.google.com/?q=13.776018,100.532719","[Roti with Chicken Curry, Beef Massaman Curry]",2,Victory Monument Thai Muslim Kitchen 51,ครัวมุสลิม อนุสาวรีย์ชัย 51,"[good_for_group, local_hidden_gem, near_bts, l...",350.00,100.00,"[halal_certified, no_pork_friendly, vegetarian...",10000000-0000-0000-0000-000000000051,"[parking_available, card_accepted]"
5,"[local, busy, casual, family_friendly]",0.624601,1.290980,64.879581,"https://maps.google.com/?q=13.744997,100.503142","[Roti with Chicken Curry, Beef Massaman Curry]",2,Chinatown Thai Muslim Kitchen 216,ครัวมุสลิม เยาวราช 216,"[tourist_area, local_market]",220.00,100.00,"[halal_certified, no_pork_friendly]",10000000-0000-0000-0000-000000000216,"[friendly_staff, parking_available, card_accep..."
6,"[local, casual, family_friendly, old_school, f...",0.624601,4.907451,60.838405,"https://maps.google.com/?q=13.780791,100.528951",[Roti with Chicken Curry],1,Ari Thai Muslim Kitchen 192,ครัวมุสลิม อารีย์ 192,"[walkable, local_hidden_gem, near_bts]",160.00,100.00,"[halal_certified, no_pork_friendly]",10000000-0000-0000-0000-000000000192,"[parking_available, card_accepted]"
7,"[local, casual, modern, family_friendly, famil...",0.624601,1.067047,60.439415,"https://maps.google.com/?q=13.742881,100.493282",[Roti with Chicken Curry],1,Chinatown Thai Muslim Kitchen 74,ครัวมุสลิม เยาวราช 74,"[good_for_group, tourist_area, local_market]",120.00,60.00,"[halal_certified, no_pork_friendly]",10000000-0000-0000-0000-000000000074,"[friendly_staff, parking_available, card_accep..."
8,"[local, old_school, casual, family_friendly]",0.561969,1.255703,57.149333,"https://maps.google.com/?q=13.750205,100.505371",[Beef Massaman Curry],1,Chinatown Thai Muslim Kitchen 40,ครัวมุสลิม เยาวราช 40,"[good_for_group, tourist_area, local_market]",300.00,50.00,"[halal_certified, no_pork_friendly]",10000000-0000-0000-0000-000000000040,"[friendly_staff, parking_available, card_accep..."
9,"[local, casual, modern, family_friendly, famil...",0.561969,5.260462,57.137435,"https

In [42]:
halal_result["restaurant_dietary_tags"].apply(lambda tags: "halal_certified" in tags).all()

np.True_

In [43]:
halal_result["service_tags"].apply(lambda tags: "parking_available" in tags).all()

np.True_

In [46]:
def compare_vector_vs_tag_search(
    query,
    user_lat,
    user_lng,
    preferred_tags,
    max_distance_km=4.0,
    budget_max=350,
    dietary_restrictions=None,
    allergies=None,
    limit=10,
):
    dietary_restrictions = dietary_restrictions or []
    allergies = allergies or []

    query_vec = model.encode([query], normalize_embeddings=True)[0].tolist()

    no_pork = "no_pork" in dietary_restrictions or "pork" in allergies
    vegetarian = "vegetarian" in dietary_restrictions
    halal_required = "halal_required" in dietary_restrictions

    allergy_filter = ""
    params = {
        "query_vec": str(query_vec),
        "user_lng": user_lng,
        "user_lat": user_lat,
        "max_distance_m": max_distance_km * 1000,
        "budget_max": budget_max,
        "preferred_tags": preferred_tags,
        "limit": limit,
    }

    if allergies:
        allergy_filter = "AND NOT (m.allergen_tags && :allergies)"
        params["allergies"] = allergies

    dietary_filter = ""
    if no_pork:
        dietary_filter += " AND 'contains_pork' <> ALL(m.dietary_tags)"
    if vegetarian:
        dietary_filter += " AND 'vegetarian_possible' = ANY(m.dietary_tags)"

    restaurant_filter = ""
    if halal_required:
        restaurant_filter += " AND 'halal_certified' = ANY(r.restaurant_dietary_tags)"

    base_cte = f"""
    WITH safe_menus AS (
        SELECT
            m.*,
            r.name_en AS restaurant_name,
            r.location,
            r.is_open,
            r.place_context_tags,
            r.atmosphere_tags,
            r.service_tags,
            r.restaurant_dietary_tags,
            r.google_maps_url,
            ST_Distance(
                r.location,
                ST_SetSRID(ST_MakePoint(:user_lng, :user_lat), 4326)::geography
            ) / 1000.0 AS distance_km
        FROM menus m
        JOIN restaurants r ON r.id = m.restaurant_id
        WHERE
            r.is_open = true
            AND m.confirmed_by_restaurant = true
            AND m.price <= :budget_max
            AND ST_DWithin(
                r.location,
                ST_SetSRID(ST_MakePoint(:user_lng, :user_lat), 4326)::geography,
                :max_distance_m
            )
            {allergy_filter}
            {dietary_filter}
            {restaurant_filter}
    )
    """

    vector_sql = base_cte + """
    SELECT
        restaurant_id,
        restaurant_name,
        name_en AS menu_name,
        food_tags,
        taste_tags,
        ingredient_tags,
        dietary_tags,
        distance_km,
        1 - (embedding_en <=> CAST(:query_vec AS vector)) AS vector_score
    FROM safe_menus
    ORDER BY embedding_en <=> CAST(:query_vec AS vector)
    LIMIT :limit;
    """

    tag_sql = base_cte + """
    SELECT
        restaurant_id,
        restaurant_name,
        name_en AS menu_name,
        food_tags,
        taste_tags,
        ingredient_tags,
        dietary_tags,
        distance_km,
        (
            SELECT COUNT(*)
            FROM unnest(
                COALESCE(food_tags, ARRAY[]::text[])
                || COALESCE(taste_tags, ARRAY[]::text[])
                || COALESCE(ingredient_tags, ARRAY[]::text[])
            ) AS tag
            WHERE tag = ANY(:preferred_tags)
        ) AS tag_score
    FROM safe_menus
    WHERE
        (
            COALESCE(food_tags, ARRAY[]::text[])
            || COALESCE(taste_tags, ARRAY[]::text[])
            || COALESCE(ingredient_tags, ARRAY[]::text[])
        ) && :preferred_tags
    ORDER BY tag_score DESC, distance_km ASC
    LIMIT :limit;
    """

    with engine.connect() as conn:
        vector_rows = conn.execute(text(vector_sql), params).mappings().all()
        tag_rows = conn.execute(text(tag_sql), params).mappings().all()

    return pd.DataFrame(vector_rows), pd.DataFrame(tag_rows)

In [47]:
query = "I want a hidden local place with bold flavors, not a tourist restaurant"

vector_df, tag_df = compare_vector_vs_tag_search(
    query=query,
    user_lat=13.7799,
    user_lng=100.5447,
    preferred_tags=["local_hidden_gem", "spicy", "isaan_food"],
    max_distance_km=4.0,
    budget_max=350,
    dietary_restrictions=[],
    allergies=[],
    limit=10,
)

print("VECTOR SEARCH RESULTS")
display(vector_df)

print("TAG-ONLY RESULTS")
display(tag_df)

VECTOR SEARCH RESULTS


,dietary_tags,distance_km,food_tags,ingredient_tags,menu_name,restaurant_id,restaurant_name,taste_tags,vector_score
0,[no_pork],1.529180,"[grilled, fish]",[fish],Grilled Tilapia,10000000-0000-0000-0000-000000000015,Ari No-Pork Isaan 15,[savory],0.270105
1,[no_pork],3.150233,"[grilled, fish]",[fish],Grilled Tilapia,10000000-0000-0000-0000-000000000033,Siam No-Pork Isaan 33,[savory],0.270105
2,"[vegetarian_possible, no_pork]",3.121389,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000220,Huai Khwang Street Snack Cart 220,[sweet],0.244020
3,"[vegetarian_possible, no_pork]",3.541797,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000207,Victory Monument Street Snack Cart 207,[sweet],0.244020
4,"[vegetarian_possible, no_pork]",1.407818,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000086,Ari Street Snack Cart 86,[sweet],0.244020
5,"[vegetarian_possible, no_pork]",2.038247,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000155,Ari Street Snack Cart 155,[sweet],0.244020
6,"[vegetarian_possible, no_pork]",3.623945,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000141,Victory Monument Street Snack Cart 141,[sweet],0.244020
7,"[vegetarian_possible, no_pork]",3.053388,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000213,Ratchathewi Street Snack Cart 213,[sweet],0.244020
8,[no_pork],3.933767,"[curry, southern_thai, fish]","[fish, turmeric]",Yellow Curry with Sea Bass,10000000-0000-0000-0000-000000000186,Ratchathewi Southern Seafood Curry 186,"[very_spicy, sour]",0.238376
9,[no_pork],2.946776,"[curry, southern_thai, fish]","[fish, turmeric]",Yellow Curry with Sea Bass,10000000-0000-0000-0000-000000000030,Victory Monument Southern Seafood Curry 30,"[very_spicy, sour]",0.238376


TAG-ONLY RESULTS


,dietary_tags,distance_km,food_tags,ingredient_tags,menu_name,restaurant_id,restaurant_name,tag_score,taste_tags
0,[no_pork],0.304105,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000126,Victory Monument Local Isaan 126,2,"[spicy, sour, salty]"
1,[contains_pork],0.304105,"[larb, isaan_food]","[pork, chili, mint]",Pork Larb,10000000-0000-0000-0000-000000000126,Victory Monument Local Isaan 126,2,"[spicy, sour]"
2,[no_pork],0.776431,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000009,Ari Local Isaan 9,2,"[spicy, sour, salty]"
3,[contains_pork],0.776431,"[larb, isaan_food]","[pork, chili, mint]",Pork Larb,10000000-0000-0000-0000-000000000009,Ari Local Isaan 9,2,"[spicy, sour]"
4,[no_pork],1.150076,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000162,Victory Monument Local Isaan 162,2,"[spicy, sour, salty]"
5,[no_pork],1.529180,"[soup, isaan_food, chicken]","[chicken, lime, chili]",Spicy Chicken Soup,10000000-0000-0000-0000-000000000015,Ari No-Pork Isaan 15,2,"[spicy, sour]"
6,[no_pork],1.592302,"[soup, isaan_food, chicken]","[chicken, lime, chili]",Spicy Chicken Soup,10000000-0000-0000-0000-000000000164,Victory Monument No-Pork Isaan 164,2,"[spicy, sour]"
7,[no_pork],1.669069,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000064,Ari Local Isaan 64,2,"[spicy, sour, salty]"
8,[contains_pork],1.961572,"[larb, isaan_food]","[pork, chili, mint]",Pork Larb,10000000-0000-0000-0000-000000000080,Ari Local Isaan 80,2,"[spicy, sour]"
9,[no_pork],1.961572,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000080,Ari Local Isaan 80,2,"[spicy, sour, salty]"


In [48]:
query = "อยากกินร้านลับ local รสจัด ไม่ใช่ร้านนักท่องเที่ยว"

vector_df_th, tag_df_th = compare_vector_vs_tag_search(
    query=query,
    user_lat=13.7799,
    user_lng=100.5447,
    preferred_tags=["local_hidden_gem", "spicy", "isaan_food"],
    max_distance_km=4.0,
    budget_max=350,
    dietary_restrictions=[],
    allergies=[],
    limit=10,
)

print("THAI QUERY — VECTOR SEARCH RESULTS")
display(vector_df_th)

print("THAI QUERY — TAG-ONLY RESULTS")
display(tag_df_th)

THAI QUERY — VECTOR SEARCH RESULTS


,dietary_tags,distance_km,food_tags,ingredient_tags,menu_name,restaurant_id,restaurant_name,taste_tags,vector_score
0,"[vegetarian_possible, no_pork]",1.407818,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000086,Ari Street Snack Cart 86,[sweet],0.283613
1,"[vegetarian_possible, no_pork]",2.038247,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000155,Ari Street Snack Cart 155,[sweet],0.283613
2,"[vegetarian_possible, no_pork]",3.121389,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000220,Huai Khwang Street Snack Cart 220,[sweet],0.283613
3,"[vegetarian_possible, no_pork]",3.541797,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000207,Victory Monument Street Snack Cart 207,[sweet],0.283613
4,"[vegetarian_possible, no_pork]",3.623945,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000141,Victory Monument Street Snack Cart 141,[sweet],0.283613
5,"[vegetarian_possible, no_pork]",3.053388,"[dessert, street_food]","[coconut, rice]",Khanom Krok,10000000-0000-0000-0000-000000000213,Ratchathewi Street Snack Cart 213,[sweet],0.283613
6,[no_pork],1.529180,"[grilled, fish]",[fish],Grilled Tilapia,10000000-0000-0000-0000-000000000015,Ari No-Pork Isaan 15,[savory],0.271427
7,[no_pork],3.150233,"[grilled, fish]",[fish],Grilled Tilapia,10000000-0000-0000-0000-000000000033,Siam No-Pork Isaan 33,[savory],0.271427
8,[contains_pork],3.087752,"[grilled, street_food]",[pork],Grilled Pork Skewers,10000000-0000-0000-0000-000000000019,Huai Khwang Street Snack Cart 19,"[sweet, savory]",0.270305
9,[contains_pork],3.623945,"[grilled, street_food]",[pork],Grilled Pork Skewers,10000000-0000-0000-0000-000000000141,Victory Monument Street Snack Cart 141,"[sweet, savory]",0.270305


THAI QUERY — TAG-ONLY RESULTS


,dietary_tags,distance_km,food_tags,ingredient_tags,menu_name,restaurant_id,restaurant_name,tag_score,taste_tags
0,[no_pork],0.304105,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000126,Victory Monument Local Isaan 126,2,"[spicy, sour, salty]"
1,[contains_pork],0.304105,"[larb, isaan_food]","[pork, chili, mint]",Pork Larb,10000000-0000-0000-0000-000000000126,Victory Monument Local Isaan 126,2,"[spicy, sour]"
2,[no_pork],0.776431,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000009,Ari Local Isaan 9,2,"[spicy, sour, salty]"
3,[contains_pork],0.776431,"[larb, isaan_food]","[pork, chili, mint]",Pork Larb,10000000-0000-0000-0000-000000000009,Ari Local Isaan 9,2,"[spicy, sour]"
4,[no_pork],1.150076,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000162,Victory Monument Local Isaan 162,2,"[spicy, sour, salty]"
5,[no_pork],1.529180,"[soup, isaan_food, chicken]","[chicken, lime, chili]",Spicy Chicken Soup,10000000-0000-0000-0000-000000000015,Ari No-Pork Isaan 15,2,"[spicy, sour]"
6,[no_pork],1.592302,"[soup, isaan_food, chicken]","[chicken, lime, chili]",Spicy Chicken Soup,10000000-0000-0000-0000-000000000164,Victory Monument No-Pork Isaan 164,2,"[spicy, sour]"
7,[no_pork],1.669069,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000064,Ari Local Isaan 64,2,"[spicy, sour, salty]"
8,[contains_pork],1.961572,"[larb, isaan_food]","[pork, chili, mint]",Pork Larb,10000000-0000-0000-0000-000000000080,Ari Local Isaan 80,2,"[spicy, sour]"
9,[no_pork],1.961572,"[som_tam, salad, isaan_food]","[papaya, crab, fermented_fish]",Som Tam Pu Pla Ra,10000000-0000-0000-0000-000000000080,Ari Local Isaan 80,2,"[spicy, sour, salty]"


In [49]:
def compare_overlap(vector_df, tag_df):
    vector_ids = set(vector_df["restaurant_id"].astype(str)) if not vector_df.empty else set()
    tag_ids = set(tag_df["restaurant_id"].astype(str)) if not tag_df.empty else set()

    overlap = vector_ids & tag_ids

    return {
        "vector_count": len(vector_ids),
        "tag_count": len(tag_ids),
        "overlap_count": len(overlap),
        "vector_unique_count": len(vector_ids - tag_ids),
        "tag_unique_count": len(tag_ids - vector_ids),
        "overlap_ratio": len(overlap) / max(1, len(vector_ids | tag_ids)),
    }

print("English query comparison:")
print(compare_overlap(vector_df, tag_df))

print("Thai query comparison:")
print(compare_overlap(vector_df_th, tag_df_th))

English query comparison:
{'vector_count': 10, 'tag_count': 7, 'overlap_count': 1, 'vector_unique_count': 9, 'tag_unique_count': 6, 'overlap_ratio': 0.0625}
Thai query comparison:
{'vector_count': 9, 'tag_count': 7, 'overlap_count': 1, 'vector_unique_count': 8, 'tag_unique_count': 6, 'overlap_ratio': 0.06666666666666667}


In [50]:
query = "somewhere auntie-style, authentic, intense flavor, not fancy"

vector_df_abstract, tag_df_abstract = compare_vector_vs_tag_search(
    query=query,
    user_lat=13.7525,
    user_lng=100.4940,
    preferred_tags=["auntie_style", "authentic", "intense_flavor"],  # intentionally not in tag catalog
    max_distance_km=5.0,
    budget_max=350,
    dietary_restrictions=[],
    allergies=[],
    limit=10,
)

print("ABSTRACT QUERY — VECTOR SEARCH RESULTS")
display(vector_df_abstract)

print("ABSTRACT QUERY — TAG-ONLY RESULTS")
display(tag_df_abstract)

ABSTRACT QUERY — VECTOR SEARCH RESULTS


,dietary_tags,distance_km,food_tags,ingredient_tags,menu_name,restaurant_id,restaurant_name,taste_tags,vector_score
0,[no_pork],1.845123,"[grilled, fish]",[fish],Grilled Tilapia,10000000-0000-0000-0000-000000000166,Chinatown No-Pork Isaan 166,[savory],0.349007
1,[no_pork],2.129510,"[grilled, fish]",[fish],Grilled Tilapia,10000000-0000-0000-0000-000000000173,Old Town No-Pork Isaan 173,[savory],0.349007
2,[no_pork],3.900347,"[curry, southern_thai, fish]","[fish, turmeric]",Yellow Curry with Sea Bass,10000000-0000-0000-0000-000000000128,Chinatown Southern Seafood Curry 128,"[very_spicy, sour]",0.299177
3,[no_pork],3.721083,"[curry, southern_thai, fish]","[fish, turmeric]",Yellow Curry with Sea Bass,10000000-0000-0000-0000-000000000183,Ari Southern Seafood Curry 183,"[very_spicy, sour]",0.299177
4,[no_pork],2.759814,"[curry, southern_thai, fish]","[fish, turmeric]",Yellow Curry with Sea Bass,10000000-0000-0000-0000-000000000172,Chinatown Southern Seafood Curry 172,"[very_spicy, sour]",0.299177
5,[no_pork],1.347959,"[curry, southern_thai, fish]","[fish, turmeric]",Yellow Curry with Sea Bass,10000000-0000-0000-0000-000000000138,Old Town Southern Seafood Curry 138,"[very_spicy, sour]",0.299177
6,[no_pork],2.747013,"[curry, southern_thai, fish]","[fish, turmeric]",Yellow Curry with Sea Bass,10000000-0000-0000-0000-000000000186,Ratchathewi Southern Seafood Curry 186,"[very_spicy, sour]",0.299177
7,[no_pork],4.371763,"[soup, isaan_food, chicken]","[chicken, lime, chili]",Spicy Chicken Soup,10000000-0000-0000-0000-000000000150,Bang Rak No-Pork Isaan 150,"[spicy, sour]",0.284577
8,[no_pork],2.129510,"[soup, isaan_food, chicken]","[chicken, lime, chili]",Spicy Chicken Soup,10000000-0000-0000-0000-000000000173,Old Town No-Pork Isaan 173,"[spicy, sour]",0.284577
9,[no_pork],4.823512,"[soup, isaan_food, chicken]","[chicken, lime, chili]",Spicy Chicken Soup,10000000-0000-0000-0000-000000000113,Silom No-Pork Isaan 113,"[spicy, sour]",0.284577


ABSTRACT QUERY — TAG-ONLY RESULTS


""
